# 🏠 Ames Housing — Pipeline Completo
## Limpeza de Dados + Feature Engineering + Regressão Linear

**Objetivo:** Preparar o dataset de Ames Housing para um modelo de regressão linear capaz de prever o preço de venda de imóveis (`SalePrice`) com a maior performance possível.

**Estrutura do notebook:**

| # | Fase | O que faz |
|---|------|-----------|
| 1 | Imports & Configuração | Bibliotecas e constantes |
| 2 | Valores Ausentes | Distingue NaN semântico de dado faltante |
| 3 | Outliers | Remove registros e transforma o target |
| 4 | Feature Engineering | Cria variáveis derivadas e interações |
| 5 | Encoding | Converte categóricas para numéricas |
| 6 | Transformação & Escala | Normaliza distribuições e escalona |
| 7 | Modelagem | Ridge, Lasso e tuning por cross-validation |
| 8 | Submissão | Predição final e exportação |

> 📌 **Contexto:** O dataset possui ~80 variáveis descrevendo atributos físicos, de localização e de condição de ~1.460 imóveis vendidos em Ames, Iowa (EUA) entre 2006 e 2010.

---
## Fase 0 — Imports e Configuração

Importamos as bibliotecas essenciais:
- **`numpy` / `pandas`**: manipulação de dados
- **`scipy.stats`**: cálculo de assimetria (skewness) das distribuições
- **`sklearn`**: pré-processamento, modelos e avaliação
- **`statsmodels`**: Variance Inflation Factor (VIF) para detectar multicolinearidade
- **`warnings`**: suprime avisos repetitivos que não afetam o resultado

In [105]:
import numpy as np
import pandas as pd
from scipy import stats

from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.linear_model import Ridge, Lasso, LinearRegression, RidgeCV, LassoCV
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from statsmodels.stats.outliers_influence import variance_inflation_factor

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
print("Bibliotecas carregadas com sucesso ✓")

Bibliotecas carregadas com sucesso ✓


---
## Carregando os Dados

Carregamos treino e teste separadamente.  
A coluna `Id` é removida (é apenas um índice) e o `SalePrice` é separado como variável target.

> ⚠️ Todo o pipeline será ajustado (**fit**) apenas no conjunto de treino e apenas **aplicado** (transform) no teste, para evitar **data leakage** — o vazamento de informação do futuro para o modelo.

In [106]:
df_train_raw = pd.read_csv("train_student.csv")
df_test_raw  = pd.read_csv("test_student.csv")

test_ids = df_test_raw["Id"].copy()

# Separar target antes de qualquer transformação
y_raw    = df_train_raw["SalePrice"].copy()
df_train = df_train_raw.drop(columns=["SalePrice", "Id"]).copy()
df_test  = df_test_raw.drop(columns=["Id"]).copy()

print(f"Treino: {df_train.shape[0]} linhas × {df_train.shape[1]} colunas")
print(f"Teste:  {df_test.shape[0]} linhas × {df_test.shape[1]} colunas")
print(f"\nDistribuição do target (SalePrice):")
print(y_raw.describe().apply(lambda x: f"${x:,.0f}"))

Treino: 1022 linhas × 79 colunas
Teste:  438 linhas × 79 colunas

Distribuição do target (SalePrice):
count      $1,022
mean     $181,313
std       $77,617
min       $34,900
25%      $130,000
50%      $165,000
75%      $215,000
max      $745,000
Name: SalePrice, dtype: str


---
## Fase 1 — Tratamento de Valores Ausentes

### Por que isso importa?
A maioria dos algoritmos não lida bem com `NaN`. Em regressão linear, um único valor ausente em uma linha faz com que toda aquela observação seja descartada por padrão.

### O ponto-chave deste dataset
Diferente do habitual, **a maioria dos `NaN` aqui não significa "dado faltante"** — ela significa **"essa propriedade não existe"**. Por exemplo:
- `FireplaceQu = NaN` → o imóvel não tem lareira
- `GarageType = NaN` → o imóvel não tem garagem
- `PoolQC = NaN` → o imóvel não tem piscina

Tratar esses casos como "dado desconhecido" e imputar com a mediana seria um **erro conceitual grave**. A estratégia correta é preencher com `"None"` (categórico) ou `0` (numérico).

### Casos que são dados realmente faltantes
- `LotFrontage`: metros de rua conectados ao lote. Faz sentido não ser zero — usamos a **mediana por bairro**, que é muito mais precisa do que a mediana global, já que lotes de um mesmo bairro tendem a ter tamanhos similares.
- `GarageYrBlt`: quando há garagem mas o ano não foi registrado, substituímos pelo `YearBuilt`.

In [107]:
def handle_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """
    Trata valores ausentes distinguindo NaN semântico (ausência da feature)
    de dado genuinamente faltante.
    """
    df = df.copy()

    # ── NaN semântico em categóricas: ausência = "não tem" ──────────────────
    # Preencher com "None" preserva a informação: o imóvel não tem aquela feature.
    # Se usássemos a moda, estaríamos inventando features que não existem.
    cat_none = [
        "PoolQC", "MiscFeature", "Alley", "Fence",
        "FireplaceQu",
        "GarageType", "GarageFinish", "GarageQual", "GarageCond",
        "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
        "MasVnrType",
    ]
    for col in cat_none:
        if col in df.columns:
            df[col] = df[col].fillna("None")

    # ── NaN semântico em numéricas: ausência = 0 ────────────────────────────
    # Ex: GarageArea=NaN significa que não há garagem, logo área = 0.
    # Imputar com mediana aqui seria incorreto (iria "inventar" uma garagem).
    num_zero = [
        "GarageArea", "GarageCars",
        "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
        "BsmtFullBath", "BsmtHalfBath",
        "MasVnrArea",
    ]
    for col in num_zero:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # ── LotFrontage: dado genuinamente faltante ──────────────────────────────
    # Imputamos com a mediana do próprio bairro (Neighborhood), pois lotes
    # do mesmo bairro tendem a ter dimensões parecidas. Muito mais preciso
    # do que a mediana global.
    if "LotFrontage" in df.columns:
        df["LotFrontage"] = df.groupby("Neighborhood")["LotFrontage"].transform(
            lambda x: x.fillna(x.median())
        )
        # Fallback: se algum bairro tiver 100% NaN, usa a mediana global
        df["LotFrontage"] = df["LotFrontage"].fillna(df["LotFrontage"].median())

    # ── GarageYrBlt: quando há garagem mas o ano não foi registrado ──────────
    # Assumir que a garagem foi construída junto com a casa é a melhor estimativa.
    if "GarageYrBlt" in df.columns:
        df["GarageYrBlt"] = df["GarageYrBlt"].fillna(df["YearBuilt"])

    # ── Variáveis com 1-2 NaN: moda ─────────────────────────────────────────
    # Com tão poucos casos, a moda é suficiente e não distorce nada.
    mode_cols = [
        "Electrical", "MSZoning", "Utilities", "Functional",
        "KitchenQual", "Exterior1st", "Exterior2nd", "SaleType",
    ]
    for col in mode_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].mode()[0])

    return df

df_train = handle_missing_values(df_train)
df_test  = handle_missing_values(df_test)

# Verificação: nenhuma coluna deve ter NaN restante (exceto as que ainda
# passarão por encoding, que serão resolvidas a seguir)
nan_restante = df_train.isnull().sum()
print("Colunas com NaN restante no treino:")
print(nan_restante[nan_restante > 0] if nan_restante.sum() > 0 else "Nenhuma ✓")

Colunas com NaN restante no treino:
Nenhuma ✓


---
## Fase 2 — Remoção de Outliers e Transformação do Target

### 2a. Outliers no conjunto de treino
O dataset contém dois registros notórios: imóveis com área habitável (`GrLivArea`) acima de 4.000 sqft, mas vendidos por valores surpreendentemente baixos. Isso ocorreu porque foram vendas parciais/atípicas — e esses pontos **puxam a reta de regressão para longe da maioria dos dados**, prejudicando muito a performance geral.

> ⚠️ Outliers são removidos **somente do treino**. No conjunto de teste, precisamos prever para todos os imóveis, independentemente de serem atípicos.

### 2b. Transformação logarítmica do target (`SalePrice`)
O `SalePrice` tem uma distribuição **assimétrica à direita** (right-skewed): há muitas casas baratas e poucas muito caras. Isso viola uma premissa da regressão linear (resíduos normalmente distribuídos) e faz o modelo subestimar preços altos.

A solução é aplicar **`log1p(SalePrice)`**, que:
1. Comprime os valores extremos, aproximando a distribuição de uma normal
2. Faz com que o modelo minimize o erro relativo (percentual) e não o erro absoluto
3. Melhora significativamente o RMSE no leaderboard do Kaggle

Na hora de prever, basta reverter com `expm1()` para recuperar os preços originais.

In [108]:
def remove_outliers(df: pd.DataFrame, target: pd.Series):
    """
    Remove os outliers documentados do dataset Ames Housing.
    Aplicar APENAS no conjunto de treino.
    
    Critério: casas com GrLivArea > 4.000 sqft e preço < $300k
    são vendas atípicas que distorcem o modelo.
    """
    mask = ~((df["GrLivArea"] > 4000) & (target < 300_000))
    return df[mask].reset_index(drop=True), target[mask].reset_index(drop=True)

def log_transform_target(target: pd.Series) -> pd.Series:
    """
    Aplica log1p no SalePrice para normalizar a distribuição assimétrica.
    log1p(x) = log(1 + x), que é seguro mesmo para x=0.
    """
    return np.log1p(target)

def inverse_transform_target(predictions: np.ndarray) -> np.ndarray:
    """
    Reverte o log1p para recuperar os preços reais nas predições finais.
    expm1(x) = exp(x) - 1, operação inversa do log1p.
    """
    return np.expm1(predictions)

n_antes = len(df_train)
df_train, y_raw = remove_outliers(df_train, y_raw)
print(f"Registros removidos como outliers: {n_antes - len(df_train)}")
print(f"Registros de treino restantes: {len(df_train)}")

# Transformação logarítmica do target
y_log = log_transform_target(y_raw)
print(f"\nSalePrice original — Skewness: {stats.skew(y_raw):.3f}")
print(f"log(SalePrice)    — Skewness: {stats.skew(y_log):.3f}  (quanto mais próximo de 0, melhor)")

Registros removidos como outliers: 2
Registros de treino restantes: 1020

SalePrice original — Skewness: 1.757
log(SalePrice)    — Skewness: 0.095  (quanto mais próximo de 0, melhor)


---
## Fase 3 — Feature Engineering

Feature engineering é o processo de **criar novas variáveis a partir das existentes** para tornar os padrões dos dados mais fáceis de capturar pelo modelo.

A regressão linear só consegue aprender relações lineares entre features e target. Ao combinarmos variáveis, estamos "mostrando" ao modelo relações que ele não conseguiria descobrir sozinho.

### Estratégias utilizadas:

**1. Combinação de áreas** — o dataset tem área do porão, 1º andar e 2º andar separadas. Somá-las em `TotalSF` cria uma medida mais representativa do tamanho total da casa.

**2. Features temporais** — o ano de construção isolado não é tão informativo quanto a **idade da casa** no momento da venda. Uma casa de 1990 vendida em 2010 tem 20 anos, o que é mais direto para o modelo.

**3. Flags binárias** — indicam a simples presença de uma feature (piscina, lareira, garagem). Muitas vezes o impacto no preço é pela existência, não pelo tamanho.

**4. Interações** — `OverallQual × GrLivArea` captura que uma casa grande E de alta qualidade vale desproporcionalmente mais do que a soma dos dois fatores separados. Essa interação é uma das features mais preditivas do dataset.

**5. Remoção de redundâncias** — após criar as features derivadas, as colunas originais que foram "consumidas" são removidas para evitar multicolinearidade.

In [109]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Cria features derivadas para capturar padrões não-lineares e interações.
    Deve ser aplicada ANTES do encoding e da escala.
    """
    df = df.copy()

    # ── 1. Áreas combinadas ──────────────────────────────────────────────────
    # Soma porão + 1º andar + 2º andar em uma única medida de tamanho total.
    # Regressão linear usa um coeficiente por feature; uma feature combinada
    # é mais eficiente do que três separadas com efeitos aditivos similares.
    df["TotalSF"] = (
        df.get("TotalBsmtSF", 0) +
        df.get("1stFlrSF", 0) +
        df.get("2ndFlrSF", 0)
    )
    # Soma todas as áreas de varanda/alpendre em uma só variável.
    df["TotalPorchSF"] = (
        df.get("OpenPorchSF", 0) + df.get("EnclosedPorch", 0) +
        df.get("3SsnPorch", 0)   + df.get("ScreenPorch", 0)
    )
    # Banheiros completos valem mais do que meios-banheiros.
    # Ponderamos 0.5 para os "half baths" (sem chuveiro/banheira).
    df["TotalBathrooms"] = (
        df.get("FullBath", 0) + 0.5 * df.get("HalfBath", 0) +
        df.get("BsmtFullBath", 0) + 0.5 * df.get("BsmtHalfBath", 0)
    )

    # ── 2. Features temporais ────────────────────────────────────────────────
    # A idade da casa (YrSold - YearBuilt) é mais diretamente interpretável
    # do que o ano absoluto de construção. Um modelo treinado em 2010 não
    # "sabe" que 1960 é antigo — mas sabe interpretar "50 anos de idade".
    yr_sold = df.get("YrSold", 2010)
    df["HouseAge"]    = (yr_sold - df["YearBuilt"]).clip(lower=0)
    df["RemodAge"]    = (yr_sold - df["YearRemodAdd"]).clip(lower=0)
    df["GarageAge"]   = (yr_sold - df.get("GarageYrBlt", df["YearBuilt"])).clip(lower=0)
    # Indicador binário: a casa foi reformada alguma vez?
    df["IsRemodeled"] = (df["YearRemodAdd"] != df["YearBuilt"]).astype(int)

    # ── 3. Flags binárias de existência ─────────────────────────────────────
    # Para features raras (ex: piscina em <10% dos imóveis), a simples presença
    # é mais informativa do que a área exata. Além disso, esses campos ficam
    # zerados para a maioria, o que pode criar padrões espúrios em modelos lineares.
    df["HasPool"]      = (df.get("PoolArea", 0) > 0).astype(int)
    df["HasGarage"]    = (df.get("GarageArea", 0) > 0).astype(int)
    df["HasBsmt"]      = (df.get("TotalBsmtSF", 0) > 0).astype(int)
    df["HasFireplace"] = (df.get("Fireplaces", 0) > 0).astype(int)
    df["Has2ndFloor"]  = (df.get("2ndFlrSF", 0) > 0).astype(int)
    df["HasAlley"]     = (df.get("Alley", "None") != "None").astype(int)

    # ── 4. Interações entre variáveis ────────────────────────────────────────
    # Regressão linear modela Y = b0 + b1*X1 + b2*X2 + ...
    # Isso implica que o efeito de X1 no preço é constante, independente de X2.
    # Mas sabemos que qualidade × área têm um efeito multiplicativo no mercado:
    # uma mansão de alta qualidade vale muito mais do que seria esperado apenas
    # somando os efeitos separados. Criar o produto X1*X2 como nova feature
    # permite que o modelo linear capture essa interação.
    df["QualxArea"]    = df["OverallQual"] * df["GrLivArea"]   # feature mais preditiva do dataset
    df["QualxAge"]     = df["OverallQual"] * df["HouseAge"]     # qualidade deprecia diferente com o tempo
    df["OverallScore"] = df["OverallQual"] * df["OverallCond"]  # combinação de qualidade e condição geral
    df["QualxTotalSF"] = df["OverallQual"] * df["TotalSF"]

    # ── 5. Remoção de colunas redundantes ────────────────────────────────────
    # Após criar features derivadas, as originais que foram "absorvidas"
    # se tornam redundantes. Mantê-las causaria multicolinearidade:
    # o modelo teria dificuldade em separar os efeitos de TotalSF vs 1stFlrSF+2ndFlrSF.
    cols_to_drop = [
        "YearBuilt", "YearRemodAdd", "GarageYrBlt",        # → substituídas pelas ages
        "1stFlrSF", "2ndFlrSF", "TotalBsmtSF",              # → substituídas por TotalSF
        "OpenPorchSF", "EnclosedPorch", "3SsnPorch",         # → substituídas por TotalPorchSF
        "ScreenPorch",
        "FullBath", "HalfBath", "BsmtFullBath", "BsmtHalfBath",  # → substituídas por TotalBathrooms
        "Utilities",    # quase sem variância: 99%+ dos imóveis têm "AllPub"
        "MoSold",       # mês de venda tem baixa importância preditiva neste dataset
    ]
    df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

    return df

df_train = engineer_features(df_train)
df_test  = engineer_features(df_test)
print(f"Features após engenharia — Treino: {df_train.shape[1]} | Teste: {df_test.shape[1]}")

Features após engenharia — Treino: 80 | Teste: 80


---
## Fase 4 — Encoding de Variáveis Categóricas

Modelos de regressão linear trabalham exclusivamente com números. Precisamos converter todas as colunas categóricas em representações numéricas, mas a estratégia varia dependendo da natureza de cada variável.

### Tipos de encoding utilizados:

**1. Mapeamento ordinal explícito** — para variáveis com ordem natural clara (qualidade, condição, etc.). Mapear `Po=1, Fa=2, TA=3, Gd=4, Ex=5` preserva a hierarquia. Usar one-hot aqui seria um erro: criaria 5 colunas binárias que "escondem" a ordem do modelo.

**2. Variáveis binárias** — `CentralAir` e `Street` já são binárias por natureza: simplesmente mapeamos para 0/1.

**3. Target encoding para `Neighborhood`** — com 25 bairros distintos, one-hot criaria 24 colunas esparsas. Target encoding substitui cada bairro pela **média do preço** das casas daquele bairro. É uma representação muito mais compacta e informativa. O risco é o **data leakage**: se calcularmos a média usando toda a coluna, o modelo "veria" o preço das casas que deveria prever. Resolvemos isso com **K-Fold out-of-fold encoding**: cada casa recebe a média calculada *sem incluir ela mesma*.

**4. One-hot encoding** — para nominais sem ordem (tipo de cobertura, fundação, etc.). Usamos `drop_first=True` para evitar a "armadilha das variáveis dummy" (multicolinearidade perfeita). O alinhamento entre treino e teste é crítico: o teste pode não ter todas as categorias, então forçamos as mesmas colunas nos dois datasets.

In [110]:
# ── Mapeamentos ordinais ────────────────────────────────────────────────────
# Escala de qualidade unificada: Po < Fa < TA < Gd < Ex
# "None" = feature ausente, recebe 0
QUALITY_MAP = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}

ORDINAL_MAPS = {
    "ExterQual":    QUALITY_MAP,
    "ExterCond":    QUALITY_MAP,
    "BsmtQual":     QUALITY_MAP,
    "BsmtCond":     QUALITY_MAP,
    "HeatingQC":    QUALITY_MAP,
    "KitchenQual":  QUALITY_MAP,
    "GarageQual":   QUALITY_MAP,
    "GarageCond":   QUALITY_MAP,
    "FireplaceQu":  QUALITY_MAP,
    "PoolQC":       QUALITY_MAP,
    # Exposição do porão ao exterior: quanto mais, mais ventilado e valorizado
    "BsmtExposure": {"None": 0, "No": 0, "Mn": 1, "Av": 2, "Gd": 3},
    # Qualidade do acabamento do porão: da pior para a melhor
    "BsmtFinType1": {"None": 0, "Unf": 0, "LwQ": 1, "Rec": 2, "BLQ": 3, "ALQ": 4, "GLQ": 5},
    "BsmtFinType2": {"None": 0, "Unf": 0, "LwQ": 1, "Rec": 2, "BLQ": 3, "ALQ": 4, "GLQ": 5},
    "GarageFinish": {"None": 0, "Unf": 0, "RFn": 1, "Fin": 2},
    "LotShape":     {"IR3": 0, "IR2": 1, "IR1": 2, "Reg": 3},
    "LandSlope":    {"Sev": 0, "Mod": 1, "Gtl": 2},
    "PavedDrive":   {"N": 0, "P": 1, "Y": 2},
    # Funcionalidade: quanto menos deduções, melhor
    "Functional":   {"Sal": 0, "Sev": 1, "Maj2": 2, "Maj1": 3, "Mod": 4, "Min2": 5, "Min1": 6, "Typ": 7},
}

BINARY_MAPS = {
    "CentralAir": {"Y": 1, "N": 0},
    "Street":     {"Pave": 1, "Grvl": 0},
}

# Nominais para one-hot (sem ordem definida)
NOMINAL_COLS = [
    "MSZoning", "LotConfig", "LandContour", "Neighborhood",
    "Condition1", "Condition2", "BldgType", "HouseStyle",
    "RoofStyle", "RoofMatl", "Exterior1st", "Exterior2nd",
    "MasVnrType", "Foundation", "Heating",
    "GarageType", "MiscFeature", "SaleType", "SaleCondition",
    "Electrical", "Alley", "Fence",
]

def encode_ordinals(df):
    df = df.copy()
    for col, mapping in ORDINAL_MAPS.items():
        if col in df.columns:
            df[col] = df[col].map(mapping).fillna(0).astype(int)
    return df

def encode_binary(df):
    df = df.copy()
    for col, mapping in BINARY_MAPS.items():
        if col in df.columns:
            df[col] = df[col].map(mapping).fillna(0).astype(int)
    return df

df_train = encode_ordinals(df_train)
df_test  = encode_ordinals(df_test)
df_train = encode_binary(df_train)
df_test  = encode_binary(df_test)
print("Encoding ordinal e binário aplicado ✓")

Encoding ordinal e binário aplicado ✓


In [ ]:
def target_encode_neighborhood(df_train, df_test, target, n_splits=5):
    """
    Target encoding do Neighborhood com K-Fold out-of-fold para evitar data leakage.

    Ideia: substituir cada bairro pela média do log(SalePrice) das casas daquele bairro.
    
    Problema do data leakage: se calcularmos a média usando TODAS as casas,
    o modelo aprende o preço médio do bairro incluindo a própria casa que está
    sendo predita — isso é "trapacear" e resulta em overfitting.

    Solução K-Fold OOF: dividimos o treino em k partes (folds). Para cada fold,
    calculamos a média do bairro usando APENAS as outras k-1 partes. Assim,
    cada casa recebe uma média calculada sem incluí-la.
    """
    df_train = df_train.copy()
    df_test  = df_test.copy()
    global_mean = target.mean()

    oof_encoded = np.zeros(len(df_train))
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    for train_idx, val_idx in kf.split(df_train):
        # Calcula média por bairro usando apenas o fold de treino
        fold_means = (
            df_train.iloc[train_idx]
            .assign(target=target.iloc[train_idx])
            .groupby("Neighborhood")["target"].mean()
        )
        # Aplica no fold de validação; bairros desconhecidos recebem a média global
        oof_encoded[val_idx] = (
            df_train.iloc[val_idx]["Neighborhood"]
            .map(fold_means).fillna(global_mean).values
        )

    df_train["Neighborhood_enc"] = oof_encoded

    # Para o teste: usa a média completa do treino (não há risco de leakage aqui,
    # pois o teste não é usado para ajustar o modelo)
    full_means = (
        df_train.assign(target=target.values)
        .groupby("Neighborhood")["target"].mean()
    )
    df_test["Neighborhood_enc"] = (
        df_test["Neighborhood"].map(full_means).fillna(global_mean)
    )

    return df_train, df_test

df_train, df_test = target_encode_neighborhood(df_train, df_test, y_log)
print("Target encoding do Neighborhood aplicado ✓")
print(f"Exemplo — médias por bairro (log scale):")
print(df_train.groupby(df_train_raw.loc[df_train.index, 'Neighborhood'] if False else
      df_train_raw.iloc[:len(df_train)]['Neighborhood'])['Neighborhood_enc']
      .mean().sort_values(ascending=False).head(5))
df_train, df_test = df_train.drop(columns=["Neighborhood"]), df_test.drop(columns=["Neighborhood"])

Target encoding do Neighborhood aplicado ✓
Exemplo — médias por bairro (log scale):
Neighborhood
NoRidge    12.188883
Somerst    12.181527
Veenker    12.166676
NridgHt    12.145444
ClearCr    12.130016
Name: Neighborhood_enc, dtype: float64


In [ ]:
def one_hot_encode(df_train, df_test):
    """
    One-hot encoding das variáveis nominais (sem ordem definida).
    
    drop_first=True: remove uma categoria de cada variável para evitar
    a 'dummy variable trap' — multicolinearidade perfeita onde a última
    categoria pode ser inferida pelas outras (é sempre 1 - soma das demais).
    
    align(): garante que treino e teste tenham exatamente as mesmas colunas.
    O teste pode não conter todas as categorias presentes no treino;
    fill_value=0 trata colunas ausentes no teste como "não pertence a essa categoria".
    """
    # Excluímos Neighborhood pois já foi tratado com target encoding acima
    nominal = [c for c in NOMINAL_COLS if c in df_train.columns and c != "Neighborhood"]

    df_train_enc = pd.get_dummies(df_train, columns=nominal, drop_first=True)
    df_test_enc  = pd.get_dummies(df_test,  columns=nominal, drop_first=True)

    df_train_enc, df_test_enc = df_train_enc.align(
        df_test_enc, join="left", axis=1, fill_value=0
    )
    return df_train_enc, df_test_enc

df_train, df_test = one_hot_encode(df_train, df_test)
print(f"One-hot encoding aplicado ✓")
print(f"Dimensões após encoding — Treino: {df_train.shape} | Teste: {df_test.shape}")

One-hot encoding aplicado ✓
Dimensões após encoding — Treino: (1020, 178) | Teste: (438, 178)


---
## Fase 5 — Transformação de Distribuições e Escala

### 5a. Log nas features assimétricas
Assim como o target, muitas features numéricas têm distribuições assimétricas (caudas longas à direita). Ex: `LotArea` — a maioria dos lotes tem entre 5.000-12.000 sqft, mas há alguns com 200.000+ sqft.

Regressão linear assume que as features têm relação linear com o target. Com distribuições muito assimétricas, os poucos valores extremos podem dominar os coeficientes. Aplicar `log1p` comprime essas caudas e aproxima a distribuição de uma normal, melhorando a qualidade dos coeficientes estimados.

Critério: aplicamos `log1p` em features com **skewness absoluta > 0,75**.

### 5b. Escala com RobustScaler
Regressão linear é sensível à escala das features. Se `GrLivArea` varia entre 300-5.000 e `OverallQual` entre 1-10, o algoritmo de otimização (gradiente) terá muito mais dificuldade, pois cada passo afeta as features de forma desigual.

Usamos **`RobustScaler`** ao invés do `StandardScaler` clássico porque:
- `StandardScaler` usa média e desvio-padrão → sensível a outliers restantes
- `RobustScaler` usa **mediana e IQR** (intervalo interquartil) → muito mais robusto

> ⚠️ **Regra fundamental:** o scaler é ajustado (`fit`) apenas no treino. O teste recebe apenas o `transform` com os parâmetros do treino. Ajustar no teste seria data leakage.

In [113]:
def log_skewed_features(df: pd.DataFrame, threshold: float = 0.75) -> pd.DataFrame:
    """
    Detecta automaticamente features numéricas com alta assimetria (skewness)
    e aplica a transformação log1p para normalizar suas distribuições.

    Threshold de 0.75 é um valor convencional na literatura de ML para
    considerar uma distribuição "significativamente assimétrica".
    Só aplicamos em colunas com valores >= 0, pois log(negativo) é indefinido.
    """
    df = df.copy()
    num_cols  = df.select_dtypes(include=[np.number]).columns.tolist()
    skewness  = df[num_cols].apply(lambda x: stats.skew(x.dropna()))
    skewed    = skewness[abs(skewness) > threshold].index.tolist()

    aplicadas = []
    for col in skewed:
        if df[col].min() >= 0:
            df[col] = np.log1p(df[col])
            aplicadas.append(col)

    print(f"log1p aplicado em {len(aplicadas)} features com |skewness| > {threshold}")
    return df

def scale_features(df_train, df_test, method="robust"):
    """
    Escala as features numéricas para equalizar suas magnitudes.

    RobustScaler (padrão): escala usando mediana e IQR.
        Fórmula: (x - mediana) / IQR
        Vantagem: resistente a outliers restantes após a limpeza.
    
    StandardScaler (alternativa): escala usando média e desvio-padrão.
        Fórmula: (x - média) / std
        Desvantagem: um único outlier extremo desloca a média e infla o std.

    IMPORTANTE: fit() apenas no treino → transform() em treino e teste.
    """
    num_cols = df_train.select_dtypes(include=[np.number]).columns.tolist()
    scaler   = RobustScaler() if method == "robust" else StandardScaler()

    df_train[num_cols] = scaler.fit_transform(df_train[num_cols])
    df_test[num_cols]  = scaler.transform(df_test[num_cols])

    return df_train, df_test

df_train = log_skewed_features(df_train)
df_test  = log_skewed_features(df_test)

df_train, df_test = scale_features(df_train, df_test, method="robust")
print("Escala aplicada com RobustScaler ✓")
print(f"\nShape final — Treino: {df_train.shape} | Teste: {df_test.shape}")

log1p aplicado em 36 features com |skewness| > 0.75
log1p aplicado em 36 features com |skewness| > 0.75
Escala aplicada com RobustScaler ✓

Shape final — Treino: (1020, 178) | Teste: (438, 178)


---
## (Opcional) Verificação de Multicolinearidade via VIF

**Multicolinearidade** ocorre quando duas ou mais features estão altamente correlacionadas entre si. Em regressão linear, isso causa:
1. Instabilidade nos coeficientes (pequenas mudanças nos dados → grandes mudanças nos pesos)
2. Dificuldade em interpretar quais features realmente impactam o target
3. Em casos extremos, a matriz não é invertível e o OLS falha

O **Variance Inflation Factor (VIF)** quantifica o quanto a variância de um coeficiente aumenta por causa da correlação com outras features:
- `VIF = 1`: sem correlação com outras features (ideal)
- `VIF < 5`: aceitável
- `VIF > 10`: problema — a feature é quase linearmente dependente de outras

> 💡 Regressão com regularização (Ridge/Lasso) é muito menos sensível à multicolinearidade do que OLS puro, pois penaliza coeficientes grandes. Ainda assim, VIF alto pode indicar features redundantes que podem ser removidas sem perda de informação.

In [114]:
def check_vif(df: pd.DataFrame, threshold: float = 10.0, max_cols: int = 50) -> pd.DataFrame:
    """
    Calcula o VIF para as features numéricas.
    Limitamos a max_cols colunas para evitar tempo de execução excessivo.
    """
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()[:max_cols]
    X = df[num_cols].dropna()

    vif_data = pd.DataFrame({
        "feature": num_cols,
        "VIF": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
    }).sort_values("VIF", ascending=False).reset_index(drop=True)

    high_vif = vif_data[vif_data["VIF"] > threshold]
    print(f"Features com VIF > {threshold} (top {len(high_vif)}):")
    print(high_vif.head(10).to_string(index=False))
    return vif_data

# Descomente para executar (pode demorar alguns minutos com muitas features):
# vif_results = check_vif(df_train)

---
## Fase 6 — Modelagem: Ridge, Lasso e Comparação

### Por que não usar OLS (Mínimos Quadrados Ordinários) puro?

Com ~200+ features após encoding, o OLS tem dois problemas:
1. **Overfitting**: com tantos parâmetros livres, o modelo memoriza o treino em vez de generalizar
2. **Instabilidade com multicolinearidade**: coeficientes explodem quando features são correlacionadas

### Ridge (L2) e Lasso (L1)

Ambos adicionam um **termo de penalidade** à função de custo do OLS:

- **OLS**: minimiza `Σ(y - ŷ)²`
- **Ridge**: minimiza `Σ(y - ŷ)² + α × Σ(w²)` → penaliza coeficientes grandes, mas não os zera
- **Lasso**: minimiza `Σ(y - ŷ)² + α × Σ|w|` → pode zerar coeficientes completamente (seleção de features)

O hiperparâmetro `α` (alpha) controla a força da regularização:
- `α → 0`: se comporta como OLS (sem regularização)
- `α → ∞`: todos os coeficientes vão a zero (modelo nulo)

### Cross-Validation
Avaliamos cada modelo com **K-Fold Cross-Validation** (k=5):
1. Dividimos o treino em 5 partes iguais
2. Treinamos em 4 partes e avaliamos na 5ª
3. Repetimos para cada fold e fazemos a média

Isso nos dá uma estimativa mais confiável da performance real do que simplesmente dividir em treino/validação uma única vez.

**Métrica:** RMSE no espaço logarítmico (equivalente ao RMSLE — Root Mean Squared Log Error), que penaliza erros proporcionais e não absolutos.

In [ ]:


def evaluate_models(X_train, y_train, cv=5):
    """
    Compara múltiplos modelos usando K-Fold cross-validation.
    Métrica: RMSE no espaço log (quanto menor, melhor).
    """
    kf = KFold(n_splits=cv, shuffle=True, random_state=42)

    models = {
        "OLS (sem regularização)":  LinearRegression(),
        "Ridge (a=1)":              Ridge(alpha=1),
        "Ridge (a=10)":             Ridge(alpha=10),
        "Ridge (a=100)":            Ridge(alpha=100),
        "Lasso (a=0.001)":          Lasso(alpha=0.001, max_iter=10_000),
        "Lasso (a=0.0001)":         Lasso(alpha=0.0001, max_iter=10_000),
    }

    print(f"{'Modelo':<30} {'RMSE médio':>12} {'± std':>10}")
    print("-" * 55)
    results = {}
    for name, model in models.items():
        scores = cross_val_score(
            model, X_train, y_train,
            scoring="neg_root_mean_squared_error", cv=kf
        )
        rmse_mean, rmse_std = -scores.mean(), scores.std()
        results[name] = rmse_mean
        print(f"{name:<30} {rmse_mean:>12.5f} {rmse_std:>10.5f}")

    best = min(results, key=results.get)
    print(f"\n→ Melhor modelo: {best} (RMSE: {results[best]:.5f})")
    def evaluate_models(X_train, y_train, cv=5):
        """
        Compara múltiplos modelos usando K-Fold cross-validation.
        Métrica: RMSE no espaço log (quanto menor, melhor).
        """
        X_train = X_train.select_dtypes(include=[np.number]).copy()

        kf = KFold(n_splits=cv, shuffle=True, random_state=42)

        models = {
            "OLS (sem regularização)":  LinearRegression(),
            "Ridge (a=1)":              Ridge(alpha=1),
            "Ridge (a=10)":             Ridge(alpha=10),
            "Ridge (a=100)":            Ridge(alpha=100),
            "Lasso (a=0.001)":          Lasso(alpha=0.001, max_iter=10_000),
            "Lasso (a=0.0001)":         Lasso(alpha=0.0001, max_iter=10_000),
        }

        print(f"{'Modelo':<30} {'RMSE médio':>12} {'± std':>10}")
        print("-" * 55)
        results = {}
        for name, model in models.items():  # Avalia cada modelo usando cross-validation e armazena os resultados em um dicionário para comparação posterior.
            scores = cross_val_score(        
                model, X_train, y_train,
                scoring="neg_root_mean_squared_error", cv=kf 
            )     
            rmse_mean, rmse_std = -scores.mean(), scores.std()
            results[name] = rmse_mean   
            print(f"{name:<30} {rmse_mean:>12.5f} {rmse_std:>10.5f}")

        best = min(results, key=results.get)
        print(f"\n→ Melhor modelo: {best} (RMSE: {results[best]:.5f})")
        return results

results = evaluate_models(df_train, y_log)

Modelo                           RMSE médio      ± std
-------------------------------------------------------


ValueError: 
All the 5 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
4 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\crisw\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\crisw\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\crisw\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_base.py", line 630, in fit
    X, y = validate_data(
           ^^^^^^^^^^^^^^
  File "c:\Users\crisw\AppData\Local\Programs\

In [ ]:
def tune_ridge(X_train, y_train, cv=5):
    """
    Busca o alpha ótimo para Ridge usando RidgeCV.
    
    RidgeCV é equivalente a testar vários alphas com cross-validation,
    mas é muito mais eficiente computacionalmente: usa a decomposição SVD
    da matriz de features para avaliar todos os alphas em uma única passagem.
    
    Testamos alphas em escala logarítmica (0.001 a 10.000) para cobrir
    uma ampla faixa sem viés para nenhuma região específica.
    """
    alphas    = np.logspace(-3, 4, 100)
    kf        = KFold(n_splits=cv, shuffle=True, random_state=42)
    ridge_cv  = RidgeCV(alphas=alphas, cv=kf, scoring="neg_root_mean_squared_error")
    ridge_cv.fit(X_train, y_train)
    print(f"Melhor alpha para Ridge: {ridge_cv.alpha_:.4f}")
    return Ridge(alpha=ridge_cv.alpha_)

def tune_lasso(X_train, y_train, cv=5):
    """
    Busca o alpha ótimo para Lasso usando LassoCV.
    
    Lasso é especialmente útil quando temos muitas features (200+):
    ele zera os coeficientes de features irrelevantes, atuando como
    um seletor automático de features. 
    
    max_iter alto é necessário pois Lasso usa coordenada descendente
    (não solução analítica como Ridge) e pode precisar de muitas iterações.
    """
    kf       = KFold(n_splits=cv, shuffle=True, random_state=42)
    lasso_cv = LassoCV(alphas=None, cv=kf, max_iter=10_000, random_state=42)
    lasso_cv.fit(X_train, y_train)
    n_selected = np.sum(lasso_cv.coef_ != 0)
    print(f"Melhor alpha para Lasso: {lasso_cv.alpha_:.6f}")
    print(f"Features selecionadas pelo Lasso: {n_selected}/{X_train.shape[1]}")
    return Lasso(alpha=lasso_cv.alpha_, max_iter=10_000)

ridge_tuned = tune_ridge(df_train, y_log)
lasso_tuned = tune_lasso(df_train, y_log)

---
## Fase 7 — Predição Final e Submissão

Treinamos o modelo escolhido em **todos os dados de treino** (sem dividir em validação) para aproveitar ao máximo as informações disponíveis.

As predições são feitas no espaço logarítmico e então revertidas com `expm1()` para obter os preços em dólares.

In [ ]:
def predict_and_submit(model, X_train, y_train, X_test, test_ids, output_path="submission.csv"):
    """
    Treina o modelo final em todos os dados de treino e gera predições.
    
    Por que treinar em todos os dados agora?
    Durante a avaliação (cross-validation), reservamos parte do treino para
    validação. Agora que já escolhemos o modelo e o alpha ideais, podemos
    usar TODOS os dados para treinar o modelo final — isso resulta em um
    modelo marginalmente melhor.
    """
    model.fit(X_train, y_train)

    # Predição no espaço log → reverter para preços reais
    predictions_log = model.predict(X_test)
    predictions     = inverse_transform_target(predictions_log)

    # Garantia de sanidade: preços não podem ser negativos
    predictions = np.maximum(predictions, 0)

    submission = pd.DataFrame({"Id": test_ids.values, "SalePrice": predictions})
    submission.to_csv(output_path, index=False)

    print(f"Submissão salva em: '{output_path}'")
    print(f"Estatísticas das predições:")
    print(f"  Mínimo:  ${predictions.min():>12,.0f}")
    print(f"  Mediana: ${np.median(predictions):>12,.0f}")
    print(f"  Média:   ${predictions.mean():>12,.0f}")
    print(f"  Máximo:  ${predictions.max():>12,.0f}")
    return submission

# Escolher o melhor modelo (ajuste conforme o resultado do evaluate_models acima)
best_model = ridge_tuned  # ou lasso_tuned

submission = predict_and_submit(
    best_model, df_train, y_log,
    df_test, test_ids,
    output_path="submission.csv",
)
submission.head(10)

---
## Resumo do Pipeline

```
Dados brutos
    │
    ├─► [Fase 1] Valores ausentes → NaN semântico (0/"None") + imputação contextual
    │
    ├─► [Fase 2] Outliers → remove 2 registros + log1p(SalePrice)
    │
    ├─► [Fase 3] Feature Engineering → TotalSF, HouseAge, QualxArea, flags binárias
    │
    ├─► [Fase 4] Encoding → ordinal explícito + target encoding + one-hot alinhado
    │
    ├─► [Fase 5] Transformação → log1p nas assimétricas + RobustScaler
    │
    └─► [Fase 6] Modelagem → Ridge/Lasso com alpha tuned por RidgeCV/LassoCV
                              → Avaliação por K-Fold cross-validation (RMSLE)
```

### Decisões-chave e suas justificativas

| Decisão | Alternativa descartada | Por quê a escolha é melhor |
|---------|----------------------|---------------------------|
| NaN semântico → 0/"None" | Imputar com mediana | Mediana inventaria features inexistentes |
| LotFrontage → mediana por bairro | Mediana global | Bairros têm tamanhos similares entre si |
| log1p(SalePrice) | Usar preço bruto | Normaliza distribuição; minimiza erro relativo |
| Mapeamento ordinal explícito | One-hot para qualidades | Preserva a hierarquia Po < Fa < TA < Gd < Ex |
| Target encoding com K-Fold | Target encoding simples | Evita data leakage nos valores de treino |
| RobustScaler | StandardScaler | Robusto a outliers restantes (usa mediana/IQR) |
| Ridge / Lasso | OLS puro | Regularização evita overfitting com 200+ features |